# VCBench Day-1 EDA

This notebook performs exploratory analysis on the full VCBench dataset. It:
- Loads the CSV
- Parses JSON fields
- Derives numeric features
- Produces the agreed visuals
- Writes a short feature-ideas summary


## 1. Setup and Load

In [ ]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path(r"C:\Users\joelb\OneDrive\Vela_partnerships_project\Project_folder")
DATA_PATH = PROJECT_ROOT / "VCBench-Starter-Kit" / "vcbench_final_public.csv"
OUT_DIR = PROJECT_ROOT / "EDA"
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
print('Rows:', len(df))
print('Columns:', df.shape[1])
df.dtypes


In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing.to_frame('missing_pct')


## 2. Outcome Balance

In [ ]:
success_counts = df['success'].value_counts().sort_index()
success_pct = df['success'].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(5, 4))
sns.barplot(x=success_counts.index.astype(str), y=success_counts.values, ax=ax)
ax.set_title('Success Label Counts')
ax.set_xlabel('success')
ax.set_ylabel('count')
for i, v in enumerate(success_counts.values):
    ax.text(i, v, f'{v} ({success_pct.iloc[i]:.1f}%)', ha='center', va='bottom')
fig.tight_layout()
fig.savefig(FIG_DIR / 'success_counts.png', dpi=150)
plt.show()

print('Class balance (%):')
print(success_pct)


## 3. Missingness Overview

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
missing.plot(kind='bar', ax=ax)
ax.set_title('Missingness by Column')
ax.set_ylabel('Missing fraction')
fig.tight_layout()
fig.savefig(FIG_DIR / 'missingness_by_column.png', dpi=150)
plt.show()

missing_cols = ['industry', 'ipos', 'acquisitions', 'educations_json', 'jobs_json']
missing[missing_cols].to_frame('missing_pct')


## 4. Categorical Overview

In [ ]:
top_n = 15
industry_counts = df['industry'].fillna('Unknown').value_counts()
top_industries = industry_counts.head(top_n)
other_count = industry_counts.iloc[top_n:].sum()
industry_plot = top_industries.copy()
if other_count > 0:
    industry_plot.loc['Other'] = other_count

fig, ax = plt.subplots(figsize=(8, 6))
industry_plot.sort_values(ascending=True).plot(kind='barh', ax=ax)
ax.set_title('Top Industries (with Other)')
ax.set_xlabel('count')
fig.tight_layout()
fig.savefig(FIG_DIR / 'industry_top.png', dpi=150)
plt.show()


In [ ]:
top_industry_labels = list(top_industries.index)
df_top = df[df['industry'].isin(top_industry_labels)].copy()
industry_success = df_top.groupby('industry')['success'].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
industry_success.plot(kind='barh', ax=ax)
ax.set_title('Success Rate by Industry (Top-N)')
ax.set_xlabel('success rate')
fig.tight_layout()
fig.savefig(FIG_DIR / 'industry_success_rate.png', dpi=150)
plt.show()


## 5. Numeric Distributions

### Definitions & Derived Fields

- `ipos_count`: number of IPO records for the founder (parsed from `ipos`).
- `acquisitions_count`: number of acquisition records for the founder (parsed from `acquisitions`).
- `ipos_count_bucket`, `acq_count_bucket`: counts binned as `0,1,2,3,4,5+`.
- `ipos_best_valuation_bucket`: highest valuation bucket across IPOs for the founder.
- `ipos_best_amount_bucket`: highest amount_raised bucket across IPOs for the founder.
- `acq_best_price_bucket`: highest acquisition price bucket across acquisitions for the founder.
- ?best bucket? means the highest value bucket by the order: `>500M`, `150M-500M`, `50M-150M`, `15M-50M`, `<15M`, `Undisclosed`, `unknown`.
- `ipos_has_any`, `acq_has_any`: 1 if any IPO/Acq record exists, else 0.
- `acq_well_known_count`: number of acquisitions marked `acquired_by_well_known = True`.
- Success-rate plots show mean success per bucket; frequency overlays show founder counts as scatter points on a log-scaled secondary axis.


In [ ]:
import ast

def parse_list_field(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    if not isinstance(x, str):
        return []
    s = x.strip()
    if s in ['', '[]']:
        return []
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return []

IPO_BUCKET_ORDER = {
    '>500M': 1,
    '150M - 500M': 2,
    '50M - 150M': 3,
    '15M - 50M': 4,
    '<15M': 5,
    'Undisclosed': 6,
    'unknown': 7
}

def best_bucket(values):
    if not values:
        return 'unknown'
    return min(values, key=lambda v: IPO_BUCKET_ORDER.get(v, 99))

ipos_rows = df['ipos'].apply(parse_list_field)
acq_rows = df['acquisitions'].apply(parse_list_field)

df['ipos_count'] = ipos_rows.apply(len)
df['acquisitions_count'] = acq_rows.apply(len)
df['ipos_has_any'] = (df['ipos_count'] > 0).astype(int)
df['acq_has_any'] = (df['acquisitions_count'] > 0).astype(int)

df['ipos_best_valuation_bucket'] = ipos_rows.apply(
    lambda rows: best_bucket([r.get('valuation_usd') for r in rows if r.get('valuation_usd')])
)
df['ipos_best_amount_bucket'] = ipos_rows.apply(
    lambda rows: best_bucket([r.get('amount_raised_usd') for r in rows if r.get('amount_raised_usd')])
)
df['acq_best_price_bucket'] = acq_rows.apply(
    lambda rows: best_bucket([r.get('price_usd') for r in rows if r.get('price_usd')])
)
df['acq_well_known_count'] = acq_rows.apply(
    lambda rows: sum(bool(r.get('acquired_by_well_known')) for r in rows)
)

df['ipos_best_valuation_score'] = df['ipos_best_valuation_bucket'].map(IPO_BUCKET_ORDER).fillna(IPO_BUCKET_ORDER['unknown'])
df['ipos_best_amount_score'] = df['ipos_best_amount_bucket'].map(IPO_BUCKET_ORDER).fillna(IPO_BUCKET_ORDER['unknown'])
df['acq_best_price_score'] = df['acq_best_price_bucket'].map(IPO_BUCKET_ORDER).fillna(IPO_BUCKET_ORDER['unknown'])

# Count distributions (0..5+)
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df['ipos_count'], bins=range(0, 7), ax=axes[0])
axes[0].set_title('IPOs Count')
axes[0].set_xlim(0, 6)
sns.histplot(df['acquisitions_count'], bins=range(0, 7), ax=axes[1])
axes[1].set_title('Acquisitions Count')
axes[1].set_xlim(0, 6)
fig.tight_layout()
fig.savefig(FIG_DIR / 'ipos_acquisitions_count_hist.png', dpi=150)
plt.show()

# Note: 'unknown' is plotted separately to keep known buckets readable.
def plot_bucket_with_unknown(counts, title_known, title_unknown, ax_known, ax_unknown):
    known_counts = counts.drop('unknown', errors='ignore')
    unknown_count = counts.get('unknown', 0)
    bucket_order = ['>500M','150M - 500M','50M - 150M','15M - 50M','<15M','Undisclosed']
    known_counts = known_counts.reindex([b for b in bucket_order if b in known_counts.index])
    known_counts.plot(kind='bar', ax=ax_known)
    ax_known.set_title(title_known)
    ax_known.set_xlabel('bucket')
    ax_known.set_ylabel('count')
    ax_unknown.bar(['unknown'], [unknown_count])
    ax_unknown.set_title(title_unknown)
    ax_unknown.set_xlabel('bucket')
    ax_unknown.set_ylabel('count')

ipos_val_counts = df['ipos_best_valuation_bucket'].value_counts()
ipos_amt_counts = df['ipos_best_amount_bucket'].value_counts()
acq_price_counts = df['acq_best_price_bucket'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_bucket_with_unknown(ipos_val_counts, 'Best IPO Valuation Bucket', 'Unknown (IPO Valuation)', axes[0], axes[1])
fig.tight_layout()
fig.savefig(FIG_DIR / 'ipos_best_valuation_bucket.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_bucket_with_unknown(ipos_amt_counts, 'Best IPO Amount Bucket', 'Unknown (IPO Amount)', axes[0], axes[1])
fig.tight_layout()
fig.savefig(FIG_DIR / 'ipos_best_amount_bucket.png', dpi=150)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_bucket_with_unknown(acq_price_counts, 'Best Acquisition Price Bucket', 'Unknown (Acq Price)', axes[0], axes[1])
fig.tight_layout()
fig.savefig(FIG_DIR / 'acq_best_price_bucket.png', dpi=150)
plt.show()

# Success rate by count buckets (0,1,2,3,4,5+) with frequency overlay
def count_bucket(x):
    if x >= 5:
        return '5+'
    return str(int(x))

df['ipos_count_bucket'] = df['ipos_count'].apply(count_bucket)
df['acq_count_bucket'] = df['acquisitions_count'].apply(count_bucket)

bucket_order = ['0','1','2','3','4','5+']

# IPO plot
ipos_rate = df.groupby('ipos_count_bucket')['success'].mean().reindex(bucket_order)
ipos_freq = df.groupby('ipos_count_bucket').size().reindex(bucket_order).fillna(0)
fig, ax1 = plt.subplots(figsize=(6, 4))
ipos_rate.plot(kind='bar', ax=ax1, color='steelblue', alpha=0.8)
ax1.set_title('IPO Count: Success Rate + Frequency')
ax1.set_xlabel('IPO count bucket')
ax1.set_ylabel('success rate')
ax2 = ax1.twinx()
ax2.scatter(bucket_order, ipos_freq.values, color='darkorange', marker='o')
ax2.set_yscale('log')
ax2.set_ylabel('founder frequency (log)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'ipos_count_success_rate_with_freq.png', dpi=150)
plt.show()

# Acquisition plot
acq_rate = df.groupby('acq_count_bucket')['success'].mean().reindex(bucket_order)
acq_freq = df.groupby('acq_count_bucket').size().reindex(bucket_order).fillna(0)
fig, ax1 = plt.subplots(figsize=(6, 4))
acq_rate.plot(kind='bar', ax=ax1, color='seagreen', alpha=0.8)
ax1.set_title('Acquisition Count: Success Rate + Frequency')
ax1.set_xlabel('Acq count bucket')
ax1.set_ylabel('success rate')
ax2 = ax1.twinx()
ax2.scatter(bucket_order, acq_freq.values, color='darkorange', marker='o')
ax2.set_yscale('log')
ax2.set_ylabel('founder frequency (log)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'acq_count_success_rate_with_freq.png', dpi=150)
plt.show()


## 6. JSON-Derived Features

### Definitions & Derived Fields

- `edu_count`: number of education entries per founder.
- `edu_any_stem`: 1 if any education field matches STEM keywords, else 0.
- `edu_best_qs_bucket`: best (lowest) QS rank bucket per founder; numeric ranks are bucketed into `<50`, `50-100`, `100-200`, `200+`.
- `job_count`: number of job entries per founder.
- `job_leadership_count`: number of roles with leadership keywords (Founder, CEO, CTO, VP, Head, Director, Lead, Chief).
- `job_large_company_count`: number of roles at large companies (>=1000 employees).
- Company size distribution is based on role counts by `company_size` bucket.


In [ ]:
STEM_KEYWORDS = [
    'engineering', 'computer', 'cs', 'mathematics', 'math', 'physics', 'statistics',
    'data science', 'biology', 'chemistry', 'electrical', 'mechanical', 'aerospace',
    'information', 'ai', 'ml'
]

LEADERSHIP_KEYWORDS = [
    'founder', 'ceo', 'cto', 'coo', 'cfo', 'vp', 'head', 'director', 'lead', 'chief'
]

LARGE_COMPANY_BUCKETS = set(['10001+ employees', '5001-10000 employees', '1000-5000 employees'])

DURATION_MAP = {
    '<2': 1,
    '2-5': 3.5,
    '5-10': 7.5,
    '10+': 12
}

QS_BUCKET_ORDER = {
    '<50': 1,
    '50-100': 2,
    '100-200': 3,
    '200+': 4,
    'unknown': 5
}

def safe_json_loads(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        return json.loads(x)
    except Exception:
        return []

def is_stem_field(field):
    if not field:
        return False
    f = field.lower()
    return any(k in f for k in STEM_KEYWORDS)

def qs_bucket(value):
    if value is None:
        return 'unknown'
    if isinstance(value, str):
        v = value.strip().lower()
        # numeric string
        if v.isdigit():
            n = int(v)
            if n < 50:
                return '<50'
            if n <= 100:
                return '50-100'
            if n <= 200:
                return '100-200'
            return '200+'
        if '1-50' in v:
            return '<50'
        if '51-100' in v or '50-100' in v:
            return '50-100'
        if '101-200' in v or '100-200' in v:
            return '100-200'
        if '200+' in v:
            return '200+'
    return 'unknown'

def best_qs_bucket(rows):
    buckets = [qs_bucket(r.get('qs_ranking')) for r in rows]
    if not buckets:
        return 'unknown'
    return min(buckets, key=lambda b: QS_BUCKET_ORDER.get(b, 99))

def leadership_role(role):
    if not role:
        return False
    r = role.lower()
    return any(k in r for k in LEADERSHIP_KEYWORDS)

edu_rows = df['educations_json'].apply(safe_json_loads)
job_rows = df['jobs_json'].apply(safe_json_loads)

edu_count = edu_rows.apply(len)
edu_any_stem = edu_rows.apply(lambda rows: int(any(is_stem_field(r.get('field')) for r in rows)))
edu_best_qs = edu_rows.apply(best_qs_bucket)

job_count = job_rows.apply(len)
job_leadership_count = job_rows.apply(lambda rows: sum(leadership_role(r.get('role')) for r in rows))
job_large_company_count = job_rows.apply(lambda rows: sum((r.get('company_size') in LARGE_COMPANY_BUCKETS) for r in rows))
job_total_duration_proxy = job_rows.apply(lambda rows: sum(DURATION_MAP.get(r.get('duration'), 0) for r in rows))

df['edu_count'] = edu_count
df['edu_any_stem'] = edu_any_stem
df['edu_best_qs_bucket'] = edu_best_qs
df['job_count'] = job_count
df['job_leadership_count'] = job_leadership_count
df['job_large_company_count'] = job_large_company_count
df['job_total_duration_proxy'] = job_total_duration_proxy

df[['edu_count','edu_any_stem','edu_best_qs_bucket','job_count','job_leadership_count','job_large_company_count','job_total_duration_proxy']].head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df['edu_count'], bins=20, ax=axes[0])
axes[0].set_title('Education Count')
sns.histplot(df['job_count'], bins=30, ax=axes[1])
axes[1].set_title('Job Count')
fig.tight_layout()
fig.savefig(FIG_DIR / 'edu_job_counts.png', dpi=150)
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
df['edu_best_qs_bucket'].value_counts().plot(kind='bar', ax=ax)
ax.set_title('Best QS Bucket')
fig.tight_layout()
fig.savefig(FIG_DIR / 'edu_qs_bucket.png', dpi=150)
plt.show()

# Company size distribution (role counts)
company_sizes = []
for rows in job_rows:
    for r in rows:
        if r.get('company_size'):
            company_sizes.append(r.get('company_size'))
company_size_counts = pd.Series(company_sizes).value_counts()
fig, ax = plt.subplots(figsize=(8, 4))
company_size_counts.plot(kind='bar', ax=ax)
ax.set_title('Company Size Distribution (Role Counts)')
ax.set_xlabel('company_size bucket')
ax.set_ylabel('role count')
fig.tight_layout()
fig.savefig(FIG_DIR / 'company_size_role_counts.png', dpi=150)
plt.show()

# Company size success rate + frequency (founder presence)
# Build founder presence per company_size bucket
company_size_buckets = sorted(company_size_counts.index.tolist())
founder_presence = {b: [] for b in company_size_buckets}
for idx, rows in enumerate(job_rows):
    sizes = set([r.get('company_size') for r in rows if r.get('company_size')])
    for b in company_size_buckets:
        founder_presence[b].append(1 if b in sizes else 0)

success_series = df['success'].reset_index(drop=True)
company_success = {b: (success_series[founder_presence[b]].mean() if sum(founder_presence[b]) > 0 else 0) for b in company_size_buckets}
company_freq = {b: sum(founder_presence[b]) for b in company_size_buckets}

fig, ax1 = plt.subplots(figsize=(8, 4))
pd.Series(company_success).plot(kind='bar', ax=ax1, color='steelblue', alpha=0.8)
ax1.set_title('Company Size: Success Rate + Frequency (Founder Presence)')
ax1.set_xlabel('company_size bucket')
ax1.set_ylabel('success rate')
ax2 = ax1.twinx()
ax2.scatter(company_size_buckets, list(company_freq.values()), color='darkorange')
ax2.set_yscale('log')
ax2.set_ylabel('founder frequency (log)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'company_size_success_rate_with_freq.png', dpi=150)
plt.show()

# Founder large company roles (count bucket)
fig, ax = plt.subplots(figsize=(6, 4))
df['job_large_company_count'].clip(upper=5).value_counts().sort_index().plot(kind='bar', ax=ax)
ax.set_title('Founder Large Company Roles (clipped at 5)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'job_large_company_count.png', dpi=150)
plt.show()

# Success rate + frequency by founder large company roles bucket
def lc_bucket(x):
    if x >= 5:
        return '5+'
    return str(int(x))
df['job_large_company_bucket'] = df['job_large_company_count'].apply(lc_bucket)
bucket_order = ['0','1','2','3','4','5+']
lc_rate = df.groupby('job_large_company_bucket')['success'].mean().reindex(bucket_order)
lc_freq = df.groupby('job_large_company_bucket').size().reindex(bucket_order).fillna(0)
fig, ax1 = plt.subplots(figsize=(6, 4))
lc_rate.plot(kind='bar', ax=ax1, color='seagreen', alpha=0.8)
ax1.set_title('Founder Large Company Roles: Success Rate + Frequency')
ax1.set_xlabel('large company roles bucket')
ax1.set_ylabel('success rate')
ax2 = ax1.twinx()
ax2.scatter(bucket_order, lc_freq.values, color='darkorange')
ax2.set_yscale('log')
ax2.set_ylabel('founder frequency (log)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'large_company_roles_success_rate_with_freq.png', dpi=150)
plt.show()


In [ ]:
qs_success = df.groupby('edu_best_qs_bucket')['success'].mean()
qs_freq = df['edu_best_qs_bucket'].value_counts()
bucket_order = ['<50','50-100','100-200','200+','unknown']
qs_success = qs_success.reindex(bucket_order)
qs_freq = qs_freq.reindex(bucket_order).fillna(0)
fig, ax1 = plt.subplots(figsize=(6, 4))
qs_success.plot(kind='bar', ax=ax1, color='steelblue', alpha=0.8)
ax1.set_title('QS Bucket: Success Rate + Frequency')
ax1.set_xlabel('QS bucket')
ax1.set_ylabel('success rate')
ax2 = ax1.twinx()
ax2.scatter(bucket_order, qs_freq.values, color='darkorange')
ax2.set_yscale('log')
ax2.set_ylabel('founder frequency (log)')
fig.tight_layout()
fig.savefig(FIG_DIR / 'qs_success_rate_with_freq.png', dpi=150)
plt.show()


## 7. Text Proxies from anonymised_prose

### Definitions & Derived Fields

- `text_char_len`: number of characters in `anonymised_prose`.
- `text_word_count`: number of whitespace-separated words in `anonymised_prose`.
- `text_sentence_count`: number of sentence-like segments (split on `.?!`).
- `text_bullet_count`: number of bullet points (lines starting with `*`).


In [ ]:
def sentence_count(text):
    if not isinstance(text, str):
        return 0
    return max(1, len(re.split(r'[.!?]+', text)) - 1)

def bullet_count(text):
    if not isinstance(text, str):
        return 0
    return len(re.findall(r'\n\*', text))

df['text_char_len'] = df['anonymised_prose'].fillna('').str.len()
df['text_word_count'] = df['anonymised_prose'].fillna('').str.split().str.len()
df['text_sentence_count'] = df['anonymised_prose'].apply(sentence_count)
df['text_bullet_count'] = df['anonymised_prose'].apply(bullet_count)

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
sns.histplot(df['text_char_len'], bins=30, ax=axes[0,0])
axes[0,0].set_title('Text Character Length')
sns.histplot(df['text_word_count'], bins=30, ax=axes[0,1])
axes[0,1].set_title('Text Word Count')
sns.histplot(df['text_sentence_count'], bins=30, ax=axes[1,0])
axes[1,0].set_title('Text Sentence Count')
sns.histplot(df['text_bullet_count'], bins=20, ax=axes[1,1])
axes[1,1].set_title('Text Bullet Count')
fig.tight_layout()
fig.savefig(FIG_DIR / 'text_proxy_hists.png', dpi=150)
plt.show()

df['text_len_quartile'] = pd.qcut(df['text_char_len'], 4, labels=False, duplicates='drop')
text_quartile_success = df.groupby('text_len_quartile')['success'].mean()
fig, ax = plt.subplots(figsize=(6, 4))
text_quartile_success.plot(kind='bar', ax=ax)
ax.set_title('Success Rate by Text Length Quartile')
ax.set_xlabel('text length quartile')
ax.set_ylabel('success rate')
fig.tight_layout()
fig.savefig(FIG_DIR / 'text_length_success_rate.png', dpi=150)
plt.show()


## 8. Correlation and Group Comparisons

### Definitions & Derived Fields

- Correlation matrix uses derived numeric features plus `success` for screening.
- Feature list is defined in the code cell below.


In [ ]:
feature_cols = [
    'success',
    'edu_count','edu_any_stem','job_count','job_leadership_count','job_large_company_count','job_total_duration_proxy',
    'text_char_len','text_word_count','text_sentence_count','text_bullet_count',
    'ipos_count','acquisitions_count','ipos_has_any','acq_has_any','acq_well_known_count',
    'ipos_best_valuation_score','ipos_best_amount_score','acq_best_price_score'
]
corr_df = df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_df, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix')
fig.tight_layout()
fig.savefig(FIG_DIR / 'correlation_matrix.png', dpi=150)
plt.show()

top_corr = corr_df['success'].drop('success').abs().sort_values(ascending=False).head(25)
top_corr.to_frame('abs_corr_with_success')


In [ ]:
group_stats = df.groupby('success')[feature_cols].agg(['mean','median'])
group_stats


## 9. Feature Ideas Notes

In [ ]:
ideas = []
ideas.append('Try composite experience score: leadership roles + large-company roles + total duration.')
ideas.append('Use education quality flags (top QS bucket) and STEM indicator as binary features.')
ideas.append('Explore industry-specific success rates: industry one-hot or target-encoded with CV.')
ideas.append('Text-length and bullet-count proxies may correlate with richer profiles; try as numeric features.')
ideas.append('Missingness in IPO/acquisition counts might carry signal; add missingness flags.')

summary_path = OUT_DIR / 'eda_summary.md'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write('# Day-1 EDA Feature Ideas\n\n')
    for i, idea in enumerate(ideas, 1):
        f.write(f'{i}. {idea}\n')

summary_path


## Verification & Logging

Lightweight checks for file existence, derived feature coverage, and output sanity.


In [ ]:
from pathlib import Path

checks = []

def log_check(name, passed, details=''):
    status = 'PASS' if passed else 'FAIL'
    checks.append((name, status, details))

# Dataset checks
required_cols = ['founder_uuid','success','industry','ipos','acquisitions','educations_json','jobs_json','anonymised_prose']
missing_cols = [c for c in required_cols if c not in df.columns]
log_check('Required columns present', len(missing_cols) == 0, f'missing={missing_cols}')
log_check('Row/col count', len(df) > 0, f'rows={len(df)}, cols={df.shape[1]}')

# Derived feature non-null counts
derived_cols = [
    'ipos_count','acquisitions_count','ipos_count_bucket','acq_count_bucket',
    'ipos_best_valuation_bucket','ipos_best_amount_bucket','acq_best_price_bucket',
    'ipos_has_any','acq_has_any','acq_well_known_count',
    'edu_count','edu_any_stem','edu_best_qs_bucket',
    'job_count','job_leadership_count','job_large_company_count','job_total_duration_proxy',
    'text_char_len','text_word_count','text_sentence_count','text_bullet_count'
]
missing_derived = [c for c in derived_cols if c not in df.columns]
log_check('Derived columns present', len(missing_derived) == 0, f'missing={missing_derived}')
if len(missing_derived) == 0:
    non_null_counts = {c: int(df[c].notna().sum()) for c in derived_cols}
    log_check('Derived non-null counts', True, str(non_null_counts))

# Output files
figs = [
    'success_counts.png',
    'missingness_by_column.png',
    'industry_top.png',
    'industry_success_rate.png',
    'ipos_acquisitions_count_hist.png',
    'ipos_best_valuation_bucket.png',
    'ipos_best_amount_bucket.png',
    'acq_best_price_bucket.png',
    'ipos_count_success_rate_with_freq.png',
    'acq_count_success_rate_with_freq.png',
    'edu_job_counts.png',
    'edu_qs_bucket.png',
    'company_size_role_counts.png',
    'job_large_company_count.png',
    'qs_success_rate.png',
    'text_proxy_hists.png',
    'text_length_success_rate.png',
    'correlation_matrix.png'
]
missing_figs = [f for f in figs if not (FIG_DIR / f).exists()]
log_check('All plot files exist', len(missing_figs) == 0, f'missing={missing_figs}')

summary_path = OUT_DIR / 'eda_summary.md'
log_check('Summary file exists', summary_path.exists(), str(summary_path))
if summary_path.exists():
    lines = summary_path.read_text(encoding='utf-8').splitlines()
    idea_count = sum(1 for line in lines if line.strip().startswith(tuple(str(i)+'.' for i in range(1, 20))))
    log_check('Summary ideas >= 5', idea_count >= 5, f'count={idea_count}')

# Print log
for name, status, details in checks:
    print(f'{status} | {name} | {details}')
